# Notebook 06: EM feature-space projections

The EM methodology extracts 21 spectral and statistical
features per trace. This notebook visualises how these features
separate normal from anomalous executions in low-dimensional
projections (PCA in 2D, t-SNE in 2D) and compares the live
reproduction against the reference figures produced by the EM
analysis pipeline.

As in notebook 05: this is a methodology demonstrator; the
numerical results here are not meant to match the paper's
reported metrics.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import IFrame, display
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

FEATURES_CSV = Path("..") / "data" / "picoc_features.csv"
FIGURES_DIR = Path("..") / "figures" / "em_evidence"

df = pd.read_csv(FEATURES_CSV)
feature_cols = ['rms', 'mean', 'std_dev', 'crest_factor', 'peak_to_peak',
                'entropy', 'peak_count', 'kurtosis', 'skewness', 'zcr',
                'energy_low', 'energy_mid', 'energy_high',
                'peak_freq', 'peak_magnitude', 'spectral_entropy',
                'spectral_crest', 'harmonic_distortion',
                'spectral_flatness', 'spectral_rolloff', 'hnr']
X = df[feature_cols].fillna(0).values
y = df["is_anomalous"].values
print(f"Feature matrix shape: {X.shape}")
print(f"Anomalous traces: {int(y.sum())}")


## Standardise features


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Mean of scaled features (should be ~0):", np.round(X_scaled.mean(axis=0), 6)[:5], "...")
print("Std of scaled features  (should be ~1):", np.round(X_scaled.std(axis=0), 6)[:5], "...")


## PCA in 2D


In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
evr = pca.explained_variance_ratio_
print(f"Explained variance ratio (PC1, PC2): {evr[0]:.3f}, {evr[1]:.3f}")
print(f"Cumulative explained variance:       {evr.sum():.3f}")

fig, ax = plt.subplots(figsize=(7, 5))
for label, colour, name in [(False, "#1f4e79", "normal"), (True, "#a23b3b", "anomalous")]:
    mask = (y == label)
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], s=8, alpha=0.5, color=colour, label=name)
ax.set_xlabel(f"PC1 ({evr[0]*100:.1f}% var)")
ax.set_ylabel(f"PC2 ({evr[1]*100:.1f}% var)")
ax.set_title("PCA projection of EM features")
ax.legend()
ax.grid(True, linestyle=":", alpha=0.5)
fig.tight_layout()
plt.show()


## t-SNE in 2D

t-SNE is initialised from the PCA solution to keep the
projection reproducible across runs.


In [ ]:
tsne = TSNE(n_components=2, perplexity=30, init="pca", random_state=42)
X_tsne = tsne.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(7, 5))
for label, colour, name in [(False, "#1f4e79", "normal"), (True, "#a23b3b", "anomalous")]:
    mask = (y == label)
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], s=8, alpha=0.5, color=colour, label=name)
ax.set_xlabel("t-SNE dim 1")
ax.set_ylabel("t-SNE dim 2")
ax.set_title("t-SNE projection of EM features (perplexity=30)")
ax.legend()
ax.grid(True, linestyle=":", alpha=0.5)
fig.tight_layout()
plt.show()


## Reference figure: PCA analysis from the EM pipeline

Reference PCA produced by the EM analysis pipeline against the
same capture. Compare with the live reproduction above; minor
visual differences are expected due to algorithmic variations
and parameter choices in the production pipeline.


In [ ]:
display(IFrame(str(FIGURES_DIR / "em_pca_analysis.pdf"), width=800, height=600))


## Reference figure: cluster comparison

Side-by-side comparison of cluster assignments produced under
different clustering settings. Used to argue that the
anomalous regime is structurally separable in feature space.


In [ ]:
display(IFrame(str(FIGURES_DIR / "em_cluster_comparison.pdf"), width=800, height=600))


## Reference figure: 3D anomaly view

Three-dimensional projection of anomalous traces against the
normal-trace background, used to visualise sub-cluster
structure inside the anomalous regime.


In [ ]:
display(IFrame(str(FIGURES_DIR / "em_anomaly_3d.pdf"), width=800, height=600))


## Reference figure: classification with zoom

Classifier decision surface in the projected feature space,
with a zoom panel onto the boundary region between normal and
anomalous traces.


In [ ]:
display(IFrame(str(FIGURES_DIR / "em_classification_zoom.pdf"), width=800, height=600))


## Reference figure: anomaly clusters

Per-cluster aggregation of the anomalous traces, exposing the
internal subdivision of the anomalous regime that motivates the
subsystem-level interpretation in the agreement step.


In [ ]:
display(IFrame(str(FIGURES_DIR / "em_anomaly_clusters.pdf"), width=800, height=600))
